# IL3.2: Análisis de Trazabilidad y Logs
## Notebook 1: Logging persistente para un agente LLM simulado y Real

### Objetivo:
Aprender por qué un agente inteligente necesita dejar registros históricos (logs) de lo que recibe, procesa y responde. Esto nos permite diagnosticar problemas, auditar decisiones y entender el comportamiento de un sistema que, de otro modo, se comportaría como una "caja negra".

### ¿Por qué hacer logging?
1. **No determinismo:** Los LLMs pueden responder diferente ante la misma entrada.
2. **Depuración:** Identificar en qué paso falló el agente (ej. fallo en API, error de parser, herramienta que no respondió).
3. **Costos y Auditoría:** Medir cuántos tokens consumimos y qué respuestas entregamos a los usuarios.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
!pip install pandas langchain langchain-openai langchain_classic wikipedia LangSmith

In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


c:\Users\realm\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ LLM de LangChain configurado.
✅ Agente y herramientas listos.


### El Problema del Agente como "Caja Negra"
Si ejecutamos un agente sin logging, no tendremos forma de saber qué pasó internamente si un usuario reporta una respuesta inesperada o errónea.

A continuación, configuraremos un sistema de logs básico con la librería estándar `logging` de Python para escribir en un archivo local llamado `agent_basic.log`.


In [4]:
import logging
from datetime import datetime

# Configuración de logging para escribir en un archivo
# Usamos el modo de escritura 'w' para reiniciar el archivo en cada ejecución de prueba, o 'a' para anexar.
logging.basicConfig(
    filename="agent_basic.log",
    filemode="w",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True  # Forzar la reconfiguración en este notebook
)

logging.info("Sistema de logging básico inicializado de forma persistente.")
print("Logging básico configurado en 'agent_basic.log'")


Logging básico configurado en 'agent_basic.log'


### Clase TraceableAgent (Agente Simulado)
Este agente no utiliza llamadas a APIs externas en su lógica interna directa. Recibe una pregunta, clasifica la intención de forma heurística, define una respuesta simulada y registra todo el proceso en el archivo de log.


In [5]:
class TraceableAgent:
    def act(self, message):
        logging.info(f"Mensaje recibido: {message}")
        
        # Lógica simple de decisión interna (Heurística)
        if "precio" in message.lower():
            intent = "consulta_comercial"
        elif "error" in message.lower():
            intent = "soporte"
        else:
            intent = "general"
        
        response = f"Intención detectada: {intent}. Respuesta generada para: {message}"
        
        # Registro de decisiones y respuestas
        logging.info(f"Intención: {intent}")
        logging.info(f"Respuesta: {response}")
        
        return response

agent = TraceableAgent()

queries = [
    "¿Cuál es el precio del servicio?",
    "Tengo un error en la plataforma",
    "Explícame qué hace este sistema"
]

print("Ejecutando consultas en el agente...")
for q in queries:
    print(f"\nConsulta: {q}")
    print(f"Respuesta: {agent.act(q)}")


Ejecutando consultas en el agente...

Consulta: ¿Cuál es el precio del servicio?
Respuesta: Intención detectada: consulta_comercial. Respuesta generada para: ¿Cuál es el precio del servicio?

Consulta: Tengo un error en la plataforma
Respuesta: Intención detectada: soporte. Respuesta generada para: Tengo un error en la plataforma

Consulta: Explícame qué hace este sistema
Respuesta: Intención detectada: general. Respuesta generada para: Explícame qué hace este sistema


### Vinculación con el LLM y Agente Real
A continuación, veremos cómo aplicar este mismo concepto de logging persistente al **LLM real de LangChain** que inicializamos al principio del notebook. 

Escribiremos una función que envuelva las llamadas al agente real (`agent_executor`) y registre en el log cuándo se inicia la llamada, qué responde el modelo y si ocurre algún error en el proceso.


In [6]:
def run_wikipedia_agent_with_logging(query):
    logging.info(f"Iniciando consulta al agente de Wikipedia. Prompt: {query}")
    print(f"Procesando consulta: '{query}'...")
    
    try:
        if llm is None:
            # Simulación en caso de que no existan las API keys configuradas en el entorno
            raise ConnectionError("El LLM no está configurado (GITHUB_TOKEN ausente). Simulación de error de red.")
            
        # Invocación real del agente de LangChain
        response = agent_executor.invoke({"input": query})
        output = response.get("output", "")
        
        logging.info(f"Respuesta del agente recibida con éxito.")
        logging.info(f"Salida del agente: {output}")
        return output
        
    except Exception as e:
        # Registrar el error con nivel ERROR en el archivo de logs
        logging.error(f"Fallo en la ejecución del agente. Motivo: {type(e).__name__} - {e}")
        print(f"❌ Ocurrió un error. Detalles en el archivo de log.")
        return f"Error al procesar: {e}"

# Ejecución 1: Consulta válida
print("--- Ejecución 1 ---")
res_real = run_wikipedia_agent_with_logging("¿Qué es la Inteligencia Artificial?")
print(f"Resultado: {res_real}\n")

# Ejecución 2: Forzar un fallo (usamos un query vacío o desconectamos simulando el error)
print("--- Ejecución 2 ---")
res_fallo = run_wikipedia_agent_with_logging("")
print(f"Resultado: {res_fallo}")


--- Ejecución 1 ---
Procesando consulta: '¿Qué es la Inteligencia Artificial?'...


> Entering new AgentExecutor chain...

Invoking: `get_wikipedia_summary` with `{'query': 'Inteligencia Artificial'}`


La inteligencia artificial, abreviada como IA o AI (por su nombre en inglés: artificial intelligence), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana.
Estas tecnologías permiten que las máquinas aprendan de la experiencia, se adapten a nuevas entradas y realicen tareas humanas como el reconocimiento de voz, la toma de decisiones, la traducción de idiomas o la visión por computadora.​​
En la actualidad, la inteligencia artificial abarca una gran variedad de subcampos.La inteligencia artificial (IA) es una disciplina de las ciencias de la computación que busca crear máq

### Lectura del archivo .log
Ahora abriremos el archivo `agent_basic.log` para verificar cómo quedó grabada la secuencia completa de eventos de ambos agentes (el simulado y el real).


In [7]:
print("--- Contenido del archivo agent_basic.log ---")
try:
    with open("agent_basic.log", "r", encoding="latin-1") as f:
        for line in f:
            print(line.strip())
except FileNotFoundError:
    print("El archivo de log no existe. Asegúrate de haber ejecutado las celdas anteriores.")


--- Contenido del archivo agent_basic.log ---
2026-06-05 19:06:03,545 | INFO | Sistema de logging básico inicializado de forma persistente.
2026-06-05 19:06:05,378 | INFO | Mensaje recibido: ¿Cuál es el precio del servicio?
2026-06-05 19:06:05,378 | INFO | Intención: consulta_comercial
2026-06-05 19:06:05,378 | INFO | Respuesta: Intención detectada: consulta_comercial. Respuesta generada para: ¿Cuál es el precio del servicio?
2026-06-05 19:06:05,378 | INFO | Mensaje recibido: Tengo un error en la plataforma
2026-06-05 19:06:05,378 | INFO | Intención: soporte
2026-06-05 19:06:05,378 | INFO | Respuesta: Intención detectada: soporte. Respuesta generada para: Tengo un error en la plataforma
2026-06-05 19:06:05,378 | INFO | Mensaje recibido: Explícame qué hace este sistema
2026-06-05 19:06:05,378 | INFO | Intención: general
2026-06-05 19:06:05,379 | INFO | Respuesta: Intención detectada: general. Respuesta generada para: Explícame qué hace este sistema
2026-06-05 19:06:09,770 | INFO | Inici

### 🛠️ Reto Práctico (Mini-entrega)

**Instrucciones:**
1. Modifica la clase `TraceableAgent` para soportar una nueva intención: `"facturacion"`. Si la consulta contiene la palabra `"factura"` o `"pago"`, la intención debe ser `"facturacion"`. Asegúrate de que esta intención se registre en el log.
2. Crea una función wrapper llamada `log_agent_run(query)` que reciba una consulta del usuario y la ejecute a través del agente de Wikipedia real `agent_executor.invoke`. La función debe registrar un mensaje de inicio, la respuesta del agente, y cualquier error intermedio en el archivo `agent_basic.log`.
3. Ejecuta la función con dos consultas y lee el log final mediante código Python para demostrar que las ejecuciones del agente real y las del simulado conviven correctamente en el mismo archivo persistente.


In [ ]:
# Desarrolla tu solución aquí

# 1. Clase TraceableAgent modificada

# 2. Función wrapper log_agent_run

# 3. Ejecución de pruebas y lectura de logs


### 📝 Preguntas de Análisis
1. **¿Cuál es la diferencia entre imprimir logs en la consola (con `print`) y escribirlos de manera persistente con la librería `logging`?**
2. **¿Qué tipo de eventos crees que son críticos registrar en producción para un agente de LangChain? (Por ejemplo: tokens consumidos, excepciones de red, tiempo de respuesta, herramientas llamadas).**
3. **Si la API de LLM responde con un error de límite de cuota (Rate Limit), ¿cómo debería el agente manejar y registrar esta situación para facilitar el soporte técnico?**
